# Ortam Yolları ve İçe Aktarmalar (Imports)

In [1]:
import sys
import logging
from pathlib import Path
from tqdm.auto import tqdm
import torch

# Proje kök dizinini ekle
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    BATCH_SIZE,
    CHROMA_PERSIST_DIR,
    DEFAULT_COMPANY_ID,
    PHOTOS_DIR,
)
from src.models import get_extractor
from src.preprocessor import ImagePreprocessor
from src.vector_store import ChromaVectorStore

# Log yapılandırması
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("production_pipeline")

COMPANY_ID = DEFAULT_COMPANY_ID
COLLECTION_NAME = "fashion_clip_products_v1" # Canlı ve temiz koleksiyon adı

print(f"✅ Ortam hazır. Hedef Firma: {COMPANY_ID} | Koleksiyon: {COLLECTION_NAME}")

✅ Ortam hazır. Hedef Firma: test_firmasi_1 | Koleksiyon: fashion_clip_products_v1


# Tüm Görselleri Tarama ve Listeleme

In [2]:
# Desteklenen formatlar
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

# Ana photos klasöründeki tüm görselleri bul
all_image_paths = [
    p for p in PHOTOS_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
]

total_images = len(all_image_paths)
print(f"📦 Toplam İşlenecek Görsel Sayısı: {total_images}")
if total_images == 0:
    raise FileNotFoundError(f"'{PHOTOS_DIR}' dizininde işlenecek görsel bulunamadı!")

📦 Toplam İşlenecek Görsel Sayısı: 3921


# CLIP Modelini ve ChromaDB'yi Başlatma

In [3]:
# 1. Şampiyon modelimizi çağırıyoruz
print("🧠 CLIP modeli belleğe yükleniyor...")
extractor = get_extractor("clip")

# 2. Görsel ön işleyici (yüz kırpma + arka plan silme)
print("🖼️ ImagePreprocessor başlatılıyor...")
preprocessor = ImagePreprocessor()

# 3. ChromaDB depomuzu başlatıyoruz
store = ChromaVectorStore(
    persist_directory=CHROMA_PERSIST_DIR,
    collection_name=COLLECTION_NAME,
    company_id=COMPANY_ID,
)

print(f"💾 ChromaDB hazır. Mevcut kayıtlı ürün sayısı: {store.count_for_tenant()}")

🧠 CLIP modeli belleğe yükleniyor...


2026-08-17 01:51:56,970 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/patrickjohncyh/fashion-clip/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-17 01:51:57,118 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/patrickjohncyh/fashion-clip/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-08-17 01:51:57,119 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-08-17 01:51:57,268 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/patrickjohncyh/fashion-clip/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-08-17 01:51:57,415 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/patrickjohncyh/fashion-clip/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-08-17 01:51:57,584 | INFO | httpx | HTTP Request: HEAD https://huggingfa

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

2026-08-17 01:51:59,503 | INFO | src.models.feature_extractor | clip hazır | model=patrickjohncyh/fashion-clip | boyut=512 | pad=True | cihaz=cpu
2026-08-17 01:51:59,505 | WARNING | src.preprocessor | OpenCV CascadeClassifier yüklenemedi, yüz kırpma atlanacak, doğrudan arka plan temizliğine geçilecek.
2026-08-17 01:51:59,607 | INFO | src.vector_store.chroma_store | ChromaDB koleksiyonu hazır | firma: test_firmasi_1 | kayıt: 0


🖼️ ImagePreprocessor başlatılıyor...
💾 ChromaDB hazır. Mevcut kayıtlı ürün sayısı: 0


# Toplu Vektör Çıkarma ve Kaydetme Döngüsü (Batch Loop + TQDM)

In [4]:
# İşlem parametreleri
batch_size = BATCH_SIZE if BATCH_SIZE else 32
total_inserted = 0

# Toplam batch sayısı
num_batches = (total_images + batch_size - 1) // batch_size

print(f"🚀 Embedding işlemi başlatılıyor... (Batch Size: {batch_size})")

# tqdm ile şık bir ilerleme çubuğu
with tqdm(total=total_images, desc="Görseller İşleniyor", unit="img") as pbar:
    for i in range(0, total_images, batch_size):
        batch_paths = all_image_paths[i : i + batch_size]

        processed_entries: list[tuple[Path, str]] = []
        for original_path in batch_paths:
            try:
                processed_image_path = preprocessor.process_image(str(original_path))
                processed_entries.append((processed_image_path, original_path.name))
            except Exception as exc:
                logger.warning(
                    "Ön işleme başarısız [%s]: %s",
                    original_path.name,
                    exc,
                )

        if not processed_entries:
            pbar.update(len(batch_paths))
            continue

        processed_paths = [entry[0] for entry in processed_entries]
        origin_by_processed_name = {
            entry[0].name: entry[1] for entry in processed_entries
        }

        try:
            # 1. Ön işlenmiş görselleri tensöre dönüştür
            tensor_batch, valid_filenames = extractor.preprocess_paths(processed_paths)

            if tensor_batch.numel() == 0 or len(valid_filenames) == 0:
                pbar.update(len(batch_paths))
                continue

            # 2. CLIP ile vektörleri çıkar
            with torch.no_grad():
                embeddings = extractor.extract(tensor_batch)
                embeddings_list = embeddings.tolist()

            # 3. ChromaDB'ye ekle — ID olarak orijinal dosya adını kullan
            chroma_ids = [origin_by_processed_name[name] for name in valid_filenames]
            inserted = store.add_embeddings(
                ids=chroma_ids,
                embeddings=embeddings_list,
            )

            total_inserted += inserted
        finally:
            for processed_image_path, _ in processed_entries:
                try:
                    processed_image_path.unlink(missing_ok=True)
                except OSError as exc:
                    logger.warning(
                        "Geçici ön işlenmiş görsel silinemedi [%s]: %s",
                        processed_image_path,
                        exc,
                    )

        pbar.update(len(batch_paths))
        pbar.set_postfix({"Başarılı Kayıt": total_inserted})

print(f"\n🎉 İşlem Tamamlandı! Toplam {total_inserted} görsel ChromaDB'ye başarıyla kaydedildi.")
print(f"📊 Veritabanındaki Güncel Kiracı Kayıt Sayısı: {store.count_for_tenant()}")

🚀 Embedding işlemi başlatılıyor... (Batch Size: 32)


Görseller İşleniyor:   0%|          | 0/3921 [00:00<?, ?img/s]

  0%|                                               | 0.00/176M [00:00<?, ?B/s]

2026-08-17 01:52:23,587 | INFO | src.preprocessor | Görsel ön işlendi | kaynak=0689.00.028.00856_0001_1.jpg | hedef=preprocessed_bee0254c55034b29aa08b8757bd74019.jpg
2026-08-17 01:52:24,571 | INFO | src.preprocessor | Görsel ön işlendi | kaynak=0309.00.026.00167_0020_1.jpg | hedef=preprocessed_4fd86992a43b4baba7e89f7304403fdf.jpg
2026-08-17 01:52:25,925 | INFO | src.preprocessor | Görsel ön işlendi | kaynak=vera-dea-5028-b-siyah.webp | hedef=preprocessed_e61d9a3d4d774fd6b68b2416da9d1cd7.jpg
2026-08-17 01:52:27,313 | INFO | src.preprocessor | Görsel ön işlendi | kaynak=ventura-2005-laci.webp | hedef=preprocessed_cfcbf959c82c4bc3b4adf6338e1115bb.jpg
2026-08-17 01:52:28,315 | INFO | src.preprocessor | Görsel ön işlendi | kaynak=0157.01.020.00503_0027_1.jpg | hedef=preprocessed_04e57440e47c4885994f16ad4b358032.jpg
2026-08-17 01:52:29,296 | INFO | src.preprocessor | Görsel ön işlendi | kaynak=womanly-2110-b-siyah.webp | hedef=preprocessed_88413818d6f84abda2facf39c1eb1a51.jpg
2026-08-17 01:5


🎉 İşlem Tamamlandı! Toplam 3921 görsel ChromaDB'ye başarıyla kaydedildi.
📊 Veritabanındaki Güncel Kiracı Kayıt Sayısı: 3921
